# LTX-Video + SANA-Sprint — T4 Optimized

**SANA-Sprint** (NVIDIA 1.6B) — генерация 1024×1024 за 1-2 шага (~0.5-1с на T4)

**LTX-Video** (Lightricks 13B) — генерация видео из текста (до 1280×720, 97 кадров)

**Оптимизации для T4 (16GB VRAM):**
- `enable_model_cpu_offload()` — последовательная выгрузка слоёв на CPU
- `vae.enable_tiling()` — VAE чанками для экономии памяти
- `enable_attention_slicing()` — sliced attention
- `torch.float16` — нативная половинная точность для T4 Tensor Cores
- `bitsandbytes 8-bit` — для text_encoder SANA-Sprint

## 0. Установка

In [ ]:
import os, sys, time, gc, torch, json
from pathlib import Path

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q accelerate transformers bitsandbytes sentencepiece ftfy
!pip install -q imageio imageio-ffmpeg safetensors huggingface_hub tqdm

DEVICE = 'cuda'
DTYPE = torch.float16  # T4 native fp16 (Tensor Cores)
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
g = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
if g: print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {g.total_memory/1e9:.1f}GB')

In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(parents=True, exist_ok=True)

def flush():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

def vr():
    a = torch.cuda.memory_allocated()/1e9
    r = torch.cuda.memory_reserved()/1e9
    return f'VRAM {a:.2f}G/{r:.2f}G'

class T:
    def __init__(self, n=''): self.n = n
    def __enter__(self): self.s = time.time(); return self
    def __exit__(self, *a): print(f'[{self.n}] {time.time()-self.s:.2f}s')

---
# 1. SANA-Sprint — Image Generation

In [ ]:
flush()
print('Loading SANA-Sprint...', vr())

from diffusers import SanaSprintPipeline, SanaSprintImg2ImgPipeline
from diffusers.models import SanaTransformer2DModel
from diffusers import BitsAndBytesConfig as DBnb
from transformers import BitsAndBytesConfig as TBnb, AutoModel

SANA_ID = 'Efficient-Large-Model/Sana_Sprint_1.6B_1024px_diffusers'

with T('SANA-Sprint load'):
    te = AutoModel.from_pretrained(SANA_ID, subfolder='text_encoder',
        quantization_config=TBnb(load_in_8bit=True),
        torch_dtype=torch.float16, device_map='auto')
    tr = SanaTransformer2DModel.from_pretrained(SANA_ID, subfolder='transformer',
        quantization_config=DBnb(load_in_8bit=True),
        torch_dtype=torch.float16, device_map='auto')
    pipe_sana = SanaSprintPipeline.from_pretrained(SANA_ID,
        text_encoder=te, transformer=tr,
        torch_dtype=torch.float16, device_map='balanced')

pipe_sana.vae.enable_tiling()
print('OK', vr())

In [ ]:
# torch.compile для ускорения transformer
if hasattr(torch, 'compile'):
    with T('compile'):
        pipe_sana.transformer = torch.compile(pipe_sana.transformer, mode='reduce-overhead')
    print('torch.compile ON')

def gen_img(prompt, h=768, w=768, steps=2, guidance=4.5, seed=42):
    flush()
    g = torch.Generator(DEVICE).manual_seed(seed) if seed else None
    with T(f'img "{prompt[:30]}..."'):
        r = pipe_sana(prompt, height=h, width=w,
                      num_inference_steps=steps, guidance_scale=guidance, generator=g)
    p = OUT / f'sana_{time.time_ns()}.png'
    r.images[0].save(p)
    print(f'{p.name} ({r.images[0].size})', vr())
    return str(p)

# Тест: 3 изображения
img1 = gen_img('Japanese garden with cherry blossoms, morning light, detailed, 4K')
img2 = gen_img('Cyberpunk city neon lights rain at night, cinematic, Blade Runner style')
img3 = gen_img('Ancient wizard casting a fire spell in an old library filled with books')

In [ ]:
# Image-to-Image с SANA-Sprint
from diffusers.utils.loading_utils import load_image

pipe_sana_i2i = SanaSprintImg2ImgPipeline.from_pretrained(SANA_ID,
    text_encoder=te, transformer=tr,
    torch_dtype=torch.float16, device_map='balanced')
pipe_sana_i2i.vae.enable_tiling()

def gen_i2i(prompt, img_path, strength=0.5, seed=77):
    flush()
    init = load_image(img_path).resize((768, 768))
    g = torch.Generator(DEVICE).manual_seed(seed)
    with T(f'i2i "{prompt[:25]}..."'):
        r = pipe_sana_i2i(prompt, image=init, strength=strength,
                         height=768, width=768, num_inference_steps=2, generator=g)
    p = OUT / f'sana_i2i_{time.time_ns()}.png'
    r.images[0].save(p)
    return str(p)

img4 = gen_i2i('A cute pink bunny, soft lighting, pastel colors', img1, 0.4)
img5 = gen_i2i('Steampunk version with bronze gears and copper', img3, 0.5)
print('I2I done', vr())

---
# 2. LTX-Video — Video Generation

In [ ]:
flush()
print('Loading LTX-Video 13B...', vr())

from diffusers import LTXPipeline
from diffusers.utils import export_to_video

with T('LTX-Video load'):
    pipe_ltx = LTXPipeline.from_pretrained(
        'Lightricks/LTX-Video', torch_dtype=torch.float16
    )

# ⚡ Ключевые оптимизации для T4:
pipe_ltx.enable_model_cpu_offload()   # последовательная выгрузка на CPU
pipe_ltx.vae.enable_tiling()          # VAE чанками
pipe_ltx.enable_attention_slicing()   # sliced attention

print('OK', vr())

In [ ]:
def gen_video(prompt, height=480, width=704, frames=49, seed=42, steps=8):
    flush()
    g = torch.Generator(DEVICE).manual_seed(seed)
    with T(f'vid "{prompt[:30]}..."'):
        r = pipe_ltx(prompt, negative_prompt='blurry, low quality, watermark',
                     width=width, height=height,
                     num_frames=frames, num_inference_steps=steps,
                     guidance_scale=3.0, generator=g)
    p = OUT / f'ltx_{time.time_ns()}.mp4'
    export_to_video(r.frames[0], str(p), fps=24)
    mb = p.stat().st_size/1e6
    print(f'{p.name} ({width}x{height}, {frames}fr, {mb:.1f}MB)', vr())
    return str(p)

# Text-to-Video
vid1 = gen_video('Sunset over calm ocean waves, golden light reflecting on water, cinematic',
                 height=480, width=704, frames=49, seed=42)
print('T2V done')

In [ ]:
# Image-to-Video с LTX-Video
from diffusers.utils import load_image

def gen_video_from_image(prompt, img_path, height=480, width=704, frames=49, seed=123):
    flush()
    init = load_image(img_path).resize((width, height))
    g = torch.Generator(DEVICE).manual_seed(seed)
    with T(f'i2v "{prompt[:30]}..."'):
        r = pipe_ltx(image=init, prompt=prompt,
                     width=width, height=height,
                     num_frames=frames, num_inference_steps=8,
                     guidance_scale=3.0, generator=g)
    p = OUT / f'ltx_i2v_{time.time_ns()}.mp4'
    export_to_video(r.frames[0], str(p), fps=24)
    mb = p.stat().st_size/1e6
    print(f'{p.name} ({width}x{height}, {frames}fr, {mb:.1f}MB)', vr())
    return str(p)

vid2 = gen_video_from_image('Gentle camera drift over a serene Japanese garden, soft sunlight',
                            img1, height=480, width=704, frames=49, seed=123)
vid3 = gen_video_from_image('Slow pan across a futuristic cyberpunk city, rain reflections, neon',
                            img2, height=480, width=704, frames=49, seed=456)
print('I2V done')

---
# 3. High Quality — Видео в 720p

In [ ]:
vid_hq = gen_video('Aerial drone shot over a misty mountain forest at sunrise, cinematic, epic',
                   height=720, width=1280, frames=65, seed=789, steps=10)
print('HQ T2V done')

---
# 4. Batch Pipeline

In [ ]:
BATCH = [
    ('image', 'Ancient Roman forum at sunrise, dramatic clouds, cinematic lighting'),
    ('image', 'Deep ocean bioluminescent jellyfish, dark blue, glowing, ethereal'),
    ('video', 'A surfer riding a massive wave at golden hour, slow motion'),
    ('video', 'Time-lapse of stars over a desert dune landscape, milky way'),
    ('image', 'Steampunk dirigible flying over Victorian London at dusk'),
    ('video', 'Candle flame flickering in a dark Gothic cathedral, dramatic shadows'),
]

batch_result = []
for i, (t, p) in enumerate(BATCH):
    print(f'\n--- [{i+1}/{len(BATCH)}] {t.upper()} ---')
    print(f'Prompt: {p}')
    try:
        if t == 'image':
            path = gen_img(p, w=768, h=768, steps=2)
        else:
            path = gen_video(p, height=480, width=704, frames=49)
        batch_result.append({'type': t, 'prompt': p, 'path': path, 'status': 'ok'})
    except Exception as e:
        print(f'FAIL: {e}')
        batch_result.append({'type': t, 'prompt': p, 'status': 'fail', 'error': str(e)})
    flush()

print(f'\nBatch: {sum(1 for r in batch_result if r["status"]=="ok")}/{len(batch_result)} OK')

---
# 5. Итоги

In [ ]:
print('='*55)
print('OUTPUT FILES')
print('='*55)
for f in sorted(OUT.iterdir()):
    mb = f.stat().st_size/1e6
    ext = f.suffix.upper()
    print(f'  {f.name:45s} {mb:7.2f} MB [{ext}]')
print('='*55)
print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
print(f'Files: {len(list(OUT.iterdir()))}')
print('\nГотово! Все файлы в /kaggle/working/output/')